# Avaliação do modelo original

Primeiro registrar o ambiente, restaurar os conjuntos verificados e definir o protocolo. A comparação deve manter o mesmo modelo-base, instruções e critérios antes e depois do ajuste. O teste fica reservado para o final.

Este notebook está em construção por etapas. A inspeção do ambiente foi executada no Colab; os valores observados estão em `reports/colab_environment.json`. Nenhum modelo foi carregado nesta primeira célula.


In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("\nPacotes instalados:")
for pacote in ("torch", "transformers", "peft", "trl", "accelerate", "bitsandbytes", "datasets"):
    try:
        print(f"  {pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"  {pacote}: não instalado")
try:
    import torch
    print("\nGPU disponível:", torch.cuda.is_available())
    print("Versão CUDA do PyTorch:", torch.version.cuda)
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_properties(0)
        print("GPU:", gpu.name)
        print(f"Memória total: {gpu.total_memory / 1024**3:.1f} GiB")
    else:
        print("Confira se o ambiente está configurado com GPU.")
except ImportError:
    print("\nPyTorch não pôde ser importado.")


## Restauração dos conjuntos

Conferir os hashes conhecidos e carregar apenas treino e validação; teste reservado.


In [ ]:
import io
import json
import hashlib
import zipfile
from pathlib import Path
from google.colab import files

PASTA = Path("/content/triunfo_data")
hashes_esperados = {
    "train": "0a139d3a1b4eb5f0d2c7f854f6e5940570769e19e053f6950290e4ee2314c003",
    "validation": "cef828a8d7ca10e0988cc16d056c571e10f13caf6c87458074bae10c2766b954",
    "test": "bf4fb9eb1316dfc621eb1b1c1085fbb44d1750b017e5eb65104240bf420f356f",
}
necessarios = ["manifest.json"] + [f"{nome}.jsonl" for nome in hashes_esperados]
if not all((PASTA / nome).is_file() for nome in necessarios):
    print("Selecione o ZIP de backup do Triunfo.")
    enviados = files.upload()
    assert len(enviados) == 1, "Selecione somente o ZIP."
    conteudo_zip = next(iter(enviados.values()))
    with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as arquivo_zip:
        assert set(arquivo_zip.namelist()) == set(necessarios)
        conteudos = {nome: arquivo_zip.read(nome) for nome in necessarios}
    for nome, esperado in hashes_esperados.items():
        observado = hashlib.sha256(conteudos[f"{nome}.jsonl"]).hexdigest()
        assert observado == esperado, f"Hash diferente em {nome}."
    PASTA.mkdir(parents=True, exist_ok=True)
    for nome, conteudo in conteudos.items():
        (PASTA / nome).write_bytes(conteudo)
    print("Backup restaurado.")
else:
    print("Arquivos encontrados; verificando integridade.")
manifesto_dados = json.loads((PASTA / "manifest.json").read_text(encoding="utf-8"))
assert manifesto_dados["dataset_revision"] == "6bda777c88d21e5a204703c1ee45597a8fa4f734"
assert manifesto_dados["seed"] == 42
for nome, esperado in hashes_esperados.items():
    observado = hashlib.sha256((PASTA / f"{nome}.jsonl").read_bytes()).hexdigest()
    assert observado == esperado, f"Hash diferente em {nome}."
    assert observado == manifesto_dados["files"][nome]["sha256"]
    print(f"{nome}: hash conferido")
def carregar_jsonl(caminho):
    with caminho.open(encoding="utf-8") as arquivo:
        return [json.loads(linha) for linha in arquivo if linha.strip()]
dados_treino = carregar_jsonl(PASTA / "train.jsonl")
dados_validacao = carregar_jsonl(PASTA / "validation.jsonl")
assert len(dados_treino) == 4469
assert len(dados_validacao) == 559
print("Treino carregado:", len(dados_treino))
print("Validação carregada:", len(dados_validacao))
print("Teste preservado no arquivo para a avaliação final.")


## Perfil de tokens no treino

Revisão fixada após a execução inicial. Medição somente no treino, preservando os gabaritos e sem truncamento. Os pesos do modelo não são carregados nesta célula.


In [ ]:
import json
import math
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False, token=False,
)
INSTRUCAO_BASE = 'You are a tool-use assistant. Use only the available tools. When calling a tool, return exactly one call inside square brackets: [tool_name(argument=value)]. Copy the tool name exactly, including spaces or dots. Use Python literal syntax for argument values. If no available tool can fulfill the request, explain why. If required information is missing, ask for it. Do not invent missing information or tool results.'

def montar_mensagens(exemplo, incluir_resposta=False):
    ferramentas = json.dumps(exemplo["ferramentas"], ensure_ascii=False, sort_keys=True)
    mensagens = [
        {"role": "system", "content": INSTRUCAO_BASE + "\n\nAvailable tools:\n" + ferramentas},
        {"role": "user", "content": exemplo["pedido"]},
    ]
    if incluir_resposta:
        mensagens.append({"role": "assistant", "content": exemplo["resposta"]})
    return mensagens

tokens_treino = []
tokens_respostas = []
for numero, exemplo in enumerate(dados_treino, start=1):
    texto = tokenizer.apply_chat_template(
        montar_mensagens(exemplo, incluir_resposta=True),
        tokenize=False, add_generation_prompt=False,
    )
    tokens_treino.append(len(tokenizer(texto, add_special_tokens=False, truncation=False)["input_ids"]))
    tokens_respostas.append(len(tokenizer(exemplo["resposta"], add_special_tokens=False, truncation=False)["input_ids"]))
    if numero % 1000 == 0:
        print(f"Medidos {numero}/{len(dados_treino)} exemplos")
def percentil(valores, proporcao):
    ordenados = sorted(valores)
    indice = max(0, math.ceil(proporcao * len(ordenados)) - 1)
    return ordenados[indice]
perfil_tokens = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "training_examples_measured": len(tokens_treino),
    "full_conversation_tokens": {
        "minimum": min(tokens_treino), "p50": percentil(tokens_treino, 0.50),
        "p90": percentil(tokens_treino, 0.90), "p95": percentil(tokens_treino, 0.95),
        "maximum": max(tokens_treino),
    },
    "reference_response_tokens": {"p95": percentil(tokens_respostas, 0.95), "maximum": max(tokens_respostas)},
    "examples_above_context_size": {
        str(limite): sum(tamanho > limite for tamanho in tokens_treino)
        for limite in (1024, 1536, 2048)
    },
    "system_instruction": INSTRUCAO_BASE,
}
pasta_execucao = Path("/content/triunfo_run")
pasta_execucao.mkdir(exist_ok=True)
(pasta_execucao / "tokenizer_profile.json").write_text(
    json.dumps(perfil_tokens, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(json.dumps(perfil_tokens, ensure_ascii=False, indent=2))


## Carregamento do modelo original

Somente no Colab/Kaggle com GPU. Mesma revisão do tokenizer, float16 e SDPA. Pesos congelados para avaliação, sem ajuste ou geração nesta célula. O tempo registrado inclui download/carregamento e não mede latência de inferência.


In [ ]:
import json
import time
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM

assert torch.cuda.is_available(), "O ambiente precisa estar com GPU."
assert MODEL_REVISION == "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
torch.cuda.reset_peak_memory_stats()
inicio = time.perf_counter()
print("Carregando o modelo original na GPU...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16,
    device_map={"": 0}, attn_implementation="sdpa",
    use_safetensors=True, trust_remote_code=False, token=False,
)
model.eval()
model.requires_grad_(False)
torch.cuda.synchronize()
registro_carregamento = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "gpu": torch.cuda.get_device_name(0),
    "dtype": str(next(model.parameters()).dtype),
    "parameters": sum(p.numel() for p in model.parameters()),
    "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "download_and_load_seconds": round(time.perf_counter() - inicio, 2),
    "gpu_allocated_gib": round(torch.cuda.memory_allocated() / 1024**3, 2),
    "gpu_peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
}
pasta_execucao = Path("/content/triunfo_run")
pasta_execucao.mkdir(exist_ok=True)
(pasta_execucao / "model_load.json").write_text(
    json.dumps(registro_carregamento, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(json.dumps(registro_carregamento, ensure_ascii=False, indent=2))


## Primeira geração do modelo original

Caso fixado pela ordem da validação. O gabarito não entra no prompt. Registrar limite de saída e EOS; não tratar o caso como estimativa de desempenho.


In [ ]:
import json
import time
from pathlib import Path
import torch
from transformers import GenerationConfig

exemplo_piloto = next(e for e in dados_validacao if e["formato"] == "chamada_unica")
mensagens = montar_mensagens(exemplo_piloto, incluir_resposta=False)
assert [m["role"] for m in mensagens] == ["system", "user"]
texto_prompt = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
entradas = tokenizer(texto_prompt, return_tensors="pt", add_special_tokens=False, truncation=False).to("cuda:0")
config_geracao = GenerationConfig(
    do_sample=False, num_beams=1, max_new_tokens=768, use_cache=True,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
torch.cuda.synchronize()
inicio = time.perf_counter()
with torch.inference_mode():
    saida = model.generate(**entradas, generation_config=config_geracao)
torch.cuda.synchronize()
tempo = time.perf_counter() - inicio
tamanho_prompt = entradas["input_ids"].shape[1]
novos_tokens = saida[0, tamanho_prompt:].tolist()
resposta_modelo = tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()
eos = config_geracao.eos_token_id
ids_fim = eos if isinstance(eos, list) else [eos]
terminou_com_eos = bool(novos_tokens) and novos_tokens[-1] in ids_fim
registro_piloto = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "example_id": exemplo_piloto["id_entrada"],
    "original_index": exemplo_piloto["indice_original"],
    "selection": "Primeiro exemplo de validação com formato chamada_unica.",
    "do_sample": False, "max_new_tokens": 768,
    "input_tokens": tamanho_prompt, "generated_tokens": len(novos_tokens),
    "first_generation_seconds": round(tempo, 2),
    "ended_with_eos": terminou_com_eos,
    "reached_token_limit": len(novos_tokens) >= 768 and not terminou_com_eos,
    "request": exemplo_piloto["pedido"], "prediction": resposta_modelo,
    "reference": exemplo_piloto["resposta"],
}
pasta_execucao = Path("/content/triunfo_run")
pasta_execucao.mkdir(exist_ok=True)
(pasta_execucao / "baseline_pilot.json").write_text(
    json.dumps(registro_piloto, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(json.dumps(registro_piloto, ensure_ascii=False, indent=2))


## Avaliador de chamadas

Mesma lógica de `src/triunfo/evaluation.py`, incluída aqui para execução no Colab sem autenticar o repositório privado. A comparação mede alinhamento ao gabarito. Texto livre não é automaticamente uma recusa correta. Os testes unitários do módulo não carregam modelos.


In [ ]:
"""Leitura estrita de chamadas; não executa ferramentas nem valida semântica."""

import ast
import json


def parse_single_call(text, tools):
    """Retorna uma chamada nomeada com valores literais ou None.

    Texto livre, chamadas múltiplas e sintaxes não suportadas retornam None.
    Isso não classifica texto livre como uma recusa correta.
    """
    text = text.strip()
    if not (text.startswith("[") and text.endswith("]")):
        return None
    interior = text[1:-1].strip()
    for tool in sorted(tools, key=lambda item: len(item["name"]), reverse=True):
        name = tool["name"]
        if not interior.startswith(name + "("):
            continue
        try:
            node = ast.parse("function" + interior[len(name):], mode="eval").body
            if not (
                isinstance(node, ast.Call)
                and isinstance(node.func, ast.Name)
                and node.func.id == "function"
            ):
                return None
            if node.args or any(argument.arg is None for argument in node.keywords):
                return None
            arguments = {
                argument.arg: ast.literal_eval(argument.value)
                for argument in node.keywords
            }
            if len(arguments) != len(node.keywords):
                return None
            json.dumps(arguments, allow_nan=False)
            return {"name": name, "arguments": arguments}
        except (SyntaxError, ValueError, TypeError, RecursionError):
            return None
    return None


def score_call(prediction, reference, tools):
    """Compara à referência; igualdade não comprova correção no mundo real.

    Ignora a ordem das chaves. Mantém comparação JSON estrita de valores,
    inclusive distinções entre números, textos e booleanos.
    """
    expected = parse_single_call(reference, tools)
    if expected is None:
        raise ValueError("Referência fora do protocolo de chamada única.")
    observed = parse_single_call(prediction, tools)
    tool_correct = observed is not None and observed["name"] == expected["name"]
    call_exact = tool_correct and json.dumps(
        observed["arguments"], sort_keys=True, ensure_ascii=False, allow_nan=False
    ) == json.dumps(
        expected["arguments"], sort_keys=True, ensure_ascii=False, allow_nan=False
    )
    return {
        "parseable_call": observed is not None,
        "tool_correct": tool_correct,
        "call_exact": call_exact,
    }

resultado_piloto = score_call(
    registro_piloto["prediction"], registro_piloto["reference"],
    exemplo_piloto["ferramentas"],
)
print(json.dumps(resultado_piloto, indent=2))


## Avaliação preliminar em 32 chamadas

Semente42, casos de validação fixados em `configs/baseline.json`. Salvar após cada exemplo e conferir protocolo antes de retomar.

A execução original no Colab concluiu32casos, mas emitiu um aviso por misturar `generation_config` e argumentos de geração. A célula consolidada abaixo usa apenas argumentos explícitos; mantém os valores fornecidos e os IDs de EOS/padding e cache da configuração usada no piloto. Essa forma corrigida ainda não foi reexecutada no Colab; o resumo observado não deve ser atribuído a uma reexecução desta revisão.

Somente chamadas únicas; resultados não são taxas de recusas corretas ou de correção no mundo real.


In [ ]:
import json
import random
import time
import hashlib
from pathlib import Path
from statistics import median
import torch

assert not any(p.requires_grad for p in model.parameters())
assert not getattr(model, "peft_config", None)
model.eval()
elegiveis = [e for e in dados_validacao if e["formato"] == "chamada_unica"]
amostra_baseline = random.Random(42).sample(elegiveis, 32)
for exemplo in amostra_baseline:
    assert parse_single_call(exemplo["resposta"], exemplo["ferramentas"]) is not None
parametros_geracao = {
    "do_sample": False, "num_beams": 1, "max_new_tokens": 768,
    "temperature": 1.0, "top_p": 1.0, "top_k": 50,
}
protocolo = {
    "phase": "base", "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "seed": 42, "example_ids": [e["id_entrada"] for e in amostra_baseline],
    "instruction_sha256": hashlib.sha256(INSTRUCAO_BASE.encode("utf-8")).hexdigest(),
    "generation": parametros_geracao,
}
caminho = Path("/content/triunfo_run/baseline_validation32.json")
caminho.parent.mkdir(exist_ok=True)
if caminho.exists():
    registro = json.loads(caminho.read_text(encoding="utf-8"))
    assert registro["protocol"] == protocolo
    resultados = registro["results"]
else:
    resultados = []
concluidos = {r["example_id"] for r in resultados}
assert len(concluidos) == len(resultados)
assert concluidos.issubset(set(protocolo["example_ids"]))
print(f"Casos já concluídos: {len(resultados)}/32")
for exemplo in amostra_baseline:
    if exemplo["id_entrada"] in concluidos:
        continue
    mensagens = montar_mensagens(exemplo, incluir_resposta=False)
    texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False, truncation=False).to("cuda:0")
    torch.cuda.synchronize()
    inicio = time.perf_counter()
    with torch.inference_mode():
        saida = model.generate(
            **entradas,
            eos_token_id=config_geracao.eos_token_id,
            pad_token_id=config_geracao.pad_token_id,
            use_cache=config_geracao.use_cache,
            **parametros_geracao,
        )
    torch.cuda.synchronize()
    segundos = time.perf_counter() - inicio
    tamanho_prompt = entradas["input_ids"].shape[1]
    gerados = saida[0, tamanho_prompt:].tolist()
    previsao = tokenizer.decode(gerados, skip_special_tokens=True).strip()
    eos = config_geracao.eos_token_id
    ids_fim = eos if isinstance(eos, list) else [eos]
    terminou_com_eos = bool(gerados) and gerados[-1] in ids_fim
    resultados.append({
        "example_id": exemplo["id_entrada"], "original_index": exemplo["indice_original"],
        "prediction": previsao, "reference": exemplo["resposta"],
        "metrics": score_call(previsao, exemplo["resposta"], exemplo["ferramentas"]),
        "input_tokens": tamanho_prompt, "generated_tokens": len(gerados),
        "generation_seconds": round(segundos, 3),
        "reached_token_limit": len(gerados) >= 768 and not terminou_com_eos,
    })
    temporario = caminho.with_suffix(".tmp")
    temporario.write_text(json.dumps({"protocol": protocolo, "results": resultados}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporario.replace(caminho)
    print(f"{len(resultados)}/32 concluídos — {segundos:.1f}s")
assert len(resultados) == 32
acertos = {nome: sum(r["metrics"][nome] for r in resultados) for nome in ("parseable_call", "tool_correct", "call_exact")}
resumo = {
    "cases": len(resultados), "counts": acertos,
    "rates": {nome: round(quantidade / len(resultados), 4) for nome, quantidade in acertos.items()},
    "reached_token_limit": sum(r["reached_token_limit"] for r in resultados),
    "median_generation_seconds": round(median(r["generation_seconds"] for r in resultados), 2),
}
print(json.dumps(resumo, indent=2))
print("Resultados salvos em:", caminho)
